# Lab 01: Logistic Regression

## Objectives:

- To learn the complete ML pipeline using Logistic Regression for Binary Classification Problem
- To train and evaluate a Logistic Regression Model 
- To compare single feature and multi feature regression models

## Theory:
### Artificial Intelligence (AI):
Artificial Intelligence (AI) can be defined as a science of making intelligent machines that can replicate human tasks and thinking.

### Machine Learning (ML):
ML is part of AI where the machines are fed data to recognize patterns and ultimately used to complete tasks on the related field.

### Deep learning (DL):
DL is a type of ML that uses neural networks to solve complex tasks/ problems. Eg. CNN models, MLPs, etc.

### Data Science:
The data processing step where data is collected, cleaned, explored, etc using DL or ML models is known as data science. 

### Supervised Learning:
Supervised Learning trains algorithms using labeled datasets. The output to the input is already known. It is used in identifying patterns and mapping inputs to the desired output.
- Regression
- Classification

### Unsupervised Learning:
Unsupervised machine learning uses algorithms to find hidden patterns, structures, or groupings in unlabeled data  without proper guidance.

### Regression:
Regression in AI is a supervised learning technique that predicts continuous numerical outcomes by analyzing the relationship between independent input variables and a dependent target variable.

### Classification:
Classification in AI is a supervised learning technique that sorts data into defined categories based on patterns learned from labeled training datasets. 

### Logistic Regression:
Logistic regression is a fundamental supervised machine learning algorithm used for binary classification.
It is a type of Classification technique.




In this assignment, we will be referring to ML Pipeline to model a logistic regression model to predict Heart Disease absence or presence. The following is the ML pipeline:

![ML pipeline to model Logistic Regression](ml_pipeline.png)

## Lab Assignment:

### Task 1: Logistic Regression using single feature

We will only be using single feature that is Cholestrol to predict whether there is presence of Heart Disease or not.

In [49]:
# Dependencies

# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt

# Machine Learning (scikit-learn)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
 accuracy_score,
 precision_score,
 recall_score,
 f1_score,
 confusion_matrix,
 classification_report
)

### **1. Data Retrieval and Collection**
The Heart Disease Dataset was loaded. The dataset provided us the following data of the 270 patients: 
- **Age**: Age of the patient,
- **Sex**: Gender (e.g. 1=male, 0=female),
- **Chest pain type**:  Type of chest pain,
- **BP**: Resting blood pressure,
- **Cholesterol**: Serum Cholestrol Level,
- **FBS over 120**: Fasting blood sugar>120 mg/dl (1 for true, 0 for false),
- **EKG results**: Resting electrocardiographic results (categorical: 0-2),
- **Max HR**: Maximum heart rate achieved,
- **Exercise angina**:  Exercise-induced angina (1 = yes, 0 = no),
- **ST depression**: ST depression induced by exercise,
- **Slope of ST**: Slope of the peak exercise ST segment (categorical: 1-3),
- **Number of vessels fluro**: Number of major vessels colored by fluoroscopy (0-3),
- **Thallium**: Thallium stress test result (categorical: 3, 6, 7),
- **Heart Disease**: Target variable (1 = Presence, 0 = Absence).  

In [50]:
data = pd.read_csv(r"Heart_Disease_Prediction.csv")
data.columns = data.columns.str.strip()
data['Heart Disease'] = data['Heart Disease'].str.strip()

In [51]:
print("Dataset Shape (rows, columns):", data.shape)
print("\nColumn Names:")
print(data.columns.tolist())

Dataset Shape (rows, columns): (270, 14)

Column Names:
['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium', 'Heart Disease']


In [52]:
# Display the first few rows of the dataset
print(data.head())

   Age  Sex  Chest pain type   BP  Cholesterol  FBS over 120  EKG results  \
0   70    1                4  130          322             0            2   
1   67    0                3  115          564             0            2   
2   57    1                2  124          261             0            0   
3   64    1                4  128          263             0            0   
4   74    0                2  120          269             0            2   

   Max HR  Exercise angina  ST depression  Slope of ST  \
0     109                0            2.4            2   
1     160                0            1.6            2   
2     141                0            0.3            1   
3     105                1            0.2            2   
4     121                1            0.2            1   

   Number of vessels fluro  Thallium Heart Disease  
0                        3         3      Presence  
1                        0         7       Absence  
2                        0   

### **2. Data Cleaning**
Data cleaning is a crucial step in AI-ML. It helps in identifying and correcting errors, inconsistencies, and inaccuracies in datasets to enhance data quality. 

Better the quality of data, closer will be the actual output to the desired output.

The following steps are generally followed during data cleaning:
- Check for missing values
- Handle missing or invalid cholesterol values
- Ensure the target variable is binary
- Verify data types

In [53]:
#Check Missing Values
print("Missing values in each column:")
data.isnull().sum()

Missing values in each column:


Age                        0
Sex                        0
Chest pain type            0
BP                         0
Cholesterol                0
FBS over 120               0
EKG results                0
Max HR                     0
Exercise angina            0
ST depression              0
Slope of ST                0
Number of vessels fluro    0
Thallium                   0
Heart Disease              0
dtype: int64

In [54]:
#Handle Invalid Cholesterol Values
#In this dataset, cholesterol value = 0 is medically invalid and treated as missing
# Ensure Cholesterol is numeric
data["Cholesterol"] = pd.to_numeric(data["Cholesterol"], errors="coerce")
# Treat invalid cholesterol values (<= 0) as missing
data.loc[data["Cholesterol"] <= 0, "Cholesterol"] = np.nan
# Fill missing cholesterol with median
data.fillna({"Cholesterol": data["Cholesterol"].median()}, inplace=True)

print("Missing Cholesterol after cleaning:", data["Cholesterol"].isna().sum())

Missing Cholesterol after cleaning: 0


### **3. Feature Design**
Feature design, also known as feature engineering, is a process of tranforming the raw data into meaningful, numeric inputs that help in better identifying the patterns in the learning process. This will ultimately help in making more accurate predictions.

Feature design removes irrelavant or redundant features to reduce noise, overfitting, and computational cost. It will also create new features from the existing ones to expose the important relationships using the domain knowledge.

To evaluate a model properly, it must be tested on a different data than the trained one. Testing and training on the same data will lead to overfitting, where the the performance of the model on the trained data will be high, while its actual performance may remain low on other data.
The simplest way to avoid this is a **train-test split method**, where the test data and the train data are splitted beforehand.

In this lab, we will be following the given steps:
- Check for missing values
- Handle missing or invalid cholesterol values
- Ensure the target variable is binary
- Verify data types

In [68]:
# Map target to numeric (0 = Absence, 1 = Presence)
if data['Heart Disease'].dtype == object or data['Heart Disease'].dtype.name == 'str':
    data['Heart Disease'] = data['Heart Disease'].map({'Absence': 0, 'Presence': 1})
data['Heart Disease']

0      1
1      0
2      1
3      0
4      0
      ..
265    0
266    0
267    0
268    0
269    1
Name: Heart Disease, Length: 270, dtype: int64

In [56]:
data[["Cholesterol", "Heart Disease"]].head()

,Cholesterol,Heart Disease
0,322.0,Presence
1,564.0,Absence
2,261.0,Presence
3,263.0,Absence
4,269.0,Absence


Since we are using only single feature Logistic Regression, columns other than 'Cholestrol' and 'Heart disease' are unnecessary data. 

Also, we updated values for column Heart Disease to contain binary values: 0 for absence and 1 for presence.

For this task, a single feature is used to predict the outcome:

- **Input Feature (X)**: Cholesterol
- **Target Variable (y)**: HeartDisease

This setup represents a simple supervised learning problem where cholesterol level is used as the sole predictor to determine whether a person has heart disease.

In [70]:
# Select single input feature
x1 = data[["Cholesterol"]] # Feature matrix (2D)
# Select target variable
y1 = data["Heart Disease"] # Target vector (1D)
print("Feature shape (X):", x1.shape)
print("Target shape (y):", y1.shape)

#Basic Sanity Check
print("First 5 feature values:")
print(x1.head())
print("\nFirst 5 target values:")
print(y1.head())

Feature shape (X): (270, 1)
Target shape (y): (270,)
First 5 feature values:
   Cholesterol
0        322.0
1        564.0
2        261.0
3        263.0
4        269.0

First 5 target values:
0    1
1    0
2    1
3    0
4    0
Name: Heart Disease, dtype: int64


Here, the feature column must be 2 dimensional because machine-learning models expect input feature to be 2-dimensional, even when we are using only one feature.

### **4. Algorithm Selection**
Logistic regression is used for heart disease prediction because the outcome is binary (presence or absence of heart disease), and logistic regression is designed to model the probability of a binary event.

Heart disease depends on many features:

Logistic regression models their **combined effect** using the log-odds:

$$
\log\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1 \cdot \text{Cholesterol} + \beta_2 \cdot \text{Age} + \dots
$$

where:

- \(p\) = probability of having heart disease  
- \(\beta_0\) = intercept  
- \(\beta_1, \beta_2, \dots\) = coefficients for each feature


### **5. Loss Function Selection**
For Logistic Regression, **Binary Cross-Entropy (Log Loss)** is used as the optimization objective as it measures how well the predicted probability matches the true label.

The loss function is defined as:

$$
L = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log(p_i) + (1 - y_i)\log(1 - p_i) \right]
$$

This loss increases significantly when the model makes confident but incorrect predictions, which makes it appropriate for binary classification tasks.

- **The more confident the wrong prediction, the bigger the penalty.**


### **6. Model Learning (Training)**
The training process involves the following steps:

- The data is divided into training and testing subsets.
- Logistic Regression is fitted using the training data.
- The model learns its parameters (weight and bias) by minimizing the log loss.
- Cholesterol values are standardized using StandardScaler to ensure more stable and efficient model training.

In [72]:

X_train, X_test, y_train, y_test = train_test_split(
 x1, y1,
 test_size=0.2,
 random_state=42,
 stratify=y1
)
print("Training set size:", X_train.shape[0])
print("Testing set size :", X_test.shape[0])

Training set size: 216
Testing set size : 54


In [73]:
model = Pipeline([
 ("scaler", StandardScaler()),
 ("lr", LogisticRegression(max_iter=2000))
])
model.fit(X_train, y_train)
print("Model trained successfully.")

Model trained successfully.


In [60]:
# Show Learned Parameters
lr = model.named_steps["lr"]
print("Coefficient (w):", lr.coef_[0][0])
print("Intercept (b):", lr.intercept_[0])

Coefficient (w): 0.17648960959078797
Intercept (b): -0.22422750268450287


Here,

**Coefficient (w):**
- If w > 0, higher cholesterol levels are associated with a higher likelihood of heart disease.
- If w < 0, an increase in cholesterol corresponds to a lower probability of heart disease.

**Intercept (b):**
Represents the baseline log-odds of heart disease when the cholesterol feature is zero (after standardization).

### **7. Model Evaluation**
The trained model is evaluated on the test set using the following metrics:

- **Accuracy**

Measures the overall correctness of the model. It shows the proportion of total predictions that are classified correctly.

- **Precision**

Indicates how many of the cases predicted as heart disease are actually correct. It helps assess the reliability of positive predictions and reduces false positives.

- **Recall (Sensitivity)**

Measures how many actual heart disease cases the model correctly identifies. It is important for ensuring that patients with heart disease are not missed.

- **F1-Score**

Combines precision and recall into a single metric. It provides a balanced measure when dealing with imbalanced datasets.

- **Confusion Matrix**

Displays the number of true positives, true negatives, false positives, and false negatives. It helps analyze where the model performs well and where it makes errors in heart disease prediction.

In [74]:
y_pred = model.predict(X_test)
print("✅ Predictions generated.")
print("First 10 predictions:", y_pred[:10])

✅ Predictions generated.
First 10 predictions: [0 0 0 0 0 0 0 0 0 0]


In [75]:
# Metrics + Confusion Matrix
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
rec = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
f1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)
cm = confusion_matrix(y_test, y_pred)
print("Accuracy :", acc)
print("Precision:", prec)
print("Recall :", rec)
print("F1-score :", f1)
print("\nConfusion Matrix:\n", cm)

Accuracy : 0.5740740740740741
Precision: 0.5714285714285714
Recall : 0.16666666666666666
F1-score : 0.25806451612903225

Confusion Matrix:
 [[27  3]
 [20  4]]


In [ ]:
# Comprehensive Classification Report
print(classification_report(y_test, y_pred, zero_division=0))

#### Understanding the Results
- **Accuracy**: Represents the overall percentage of correct predictions made by the model.  
- **Precision**: Indicates what fraction of patients predicted to have heart disease actually have the condition.  
- **Recall**: Shows how many true heart disease cases were successfully detected by the model (critical in medical applications).  
- **F1-score**: Provides a balanced metric combining both precision and recall.  

**Confusion Matrix Breakdown**:
- **True Positives (TP)**: Heart disease cases correctly predicted by the model  
- **True Negatives (TN)**: Healthy patients correctly identified as not having heart disease  
- **False Positives (FP)**: Healthy patients incorrectly predicted as having heart disease  
- **False Negatives (FN)**: Heart disease cases that the model failed to detect (most critical errors in medical diagnosis)

In [ ]:
# Visualizing the Sigmoid Curve

scaler = model.named_steps["scaler"]
lr = model.named_steps["lr"]

# Generate a smooth range of cholesterol values
chol_min = x1["Cholesterol"].min()
chol_max = x1["Cholesterol"].max()
chol_range = np.linspace(chol_min, chol_max, 400).reshape(-1, 1)

# Apply the trained scaler transformation
chol_scaled = scaler.transform(chol_range)

# Calculate probability of heart disease (class 1)
prob_heart = lr.predict_proba(chol_scaled)[:, 1]

# Create the plot
plt.figure(figsize=(8, 5))
plt.plot(chol_range, prob_heart, color='blue', linewidth=2)
plt.xlabel("Cholesterol Level")
plt.ylabel("Probability of Heart Disease")
plt.title("Logistic Regression Sigmoid Curve: Single Feature (Cholesterol)")
plt.grid(True, alpha=0.3)
plt.show()

### Interpreting the Sigmoid Curve

- The sigmoid curve demonstrates the relationship between cholesterol levels and the predicted likelihood of heart disease.
- Higher cholesterol values result in an increased predicted probability of heart disease.
- Lower cholesterol levels correspond to reduced risk according to the model.
- The characteristic S-shaped curve shows how logistic regression maps input values to probabilities bounded between 0 and 1.
- This visualization confirms that cholesterol serves as a contributing factor in assessing cardiovascular risk.

---

## Task 2: Logistic Regression using Multiple Features

In this section, we will develop a more robust model by utilizing **all available input features** to predict heart disease. Incorporating multiple features enables the model to identify complex patterns and relationships that cannot be captured by a single predictor.

### 1. Data Retrieval and Verification

We will continue using the same dataset that was loaded and preprocessed in Task 1. The data cleaning steps have already been completed.

In [ ]:
# Confirm the data is ready for multi-feature modeling
print("Dataset Dimensions:", data.shape)
print("\nPreview of the data:")
print(data.head())
print("\nColumn data types:")
print(data.dtypes)

### 2. Data Quality Check

The dataset has been cleaned during Task 1:
- Invalid cholesterol values were addressed
- The target variable was converted to binary format (0 and 1)
- Missing values were handled appropriately

Now we verify the data is ready for multi-feature analysis.

In [ ]:
# Verify no missing values remain
print("Missing values per column:")
print(data.isnull().sum())
print("\n" + "="*50)
print("\nData type overview:")
print(data.dtypes)

### 3. Feature Engineering for Multi-Feature Model

In this step, we will:
1. **Extract all input features** (excluding the target column)
2. **Verify feature encoding** is appropriate for logistic regression
3. **Apply standardization** to ensure uniform contribution from all features

#### Feature Categories

The dataset includes:
- **Continuous features**: Age, BP, Cholesterol, Max HR, ST depression
- **Categorical/Binary features**: Sex, Chest pain type, FBS over 120, EKG results, Exercise angina, Slope of ST, Number of vessels fluro, Thallium

Since all features are already numerically encoded, they can be directly used with logistic regression.

In [ ]:
# Define features and target for multi-feature model
X_multi = data.drop(columns=['Heart Disease'])
y_multi = data['Heart Disease']

print("Multi-feature input dimensions (X):", X_multi.shape)
print("Target variable dimensions (y):", y_multi.shape)
print("\nFeatures included in the model:")
print(X_multi.columns.tolist())
print("\nSample of the feature matrix:")
print(X_multi.head())

In [ ]:
# Examine unique values for each feature
print("Unique value counts per feature:\n")
for col in X_multi.columns:
    unique_count = X_multi[col].nunique()
    print(f"{col}: {unique_count} unique values")
    if unique_count <= 10:
        print(f"  Values: {sorted(X_multi[col].unique())}")
    print()

#### Splitting Data for Training and Testing

Similar to Task 1, we partition the data into training and testing subsets to evaluate model performance on unseen data.

In [ ]:
# Create train-test split for multi-feature model
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_multi, y_multi,
    test_size=0.2,
    random_state=42,
    stratify=y_multi
)

print("Training samples:", X_train_multi.shape[0])
print("Testing samples:", X_test_multi.shape[0])
print("\nTraining feature dimensions:", X_train_multi.shape)
print("Testing feature dimensions:", X_test_multi.shape)

### Benefits of Using Multiple Features

Employing multiple features rather than a single predictor provides several advantages:

1. **Capturing Complex Relationships**: Heart disease is influenced by numerous risk factors (age, blood pressure, cholesterol, etc.). A multi-feature model can learn how these variables interact.

2. **Minimizing Information Loss**: Depending solely on cholesterol ignores other crucial predictors such as age, exercise tolerance, and chest pain characteristics.

3. **Enhanced Predictive Capability**: Additional relevant features provide richer context, resulting in improved accuracy and recall.

4. **Improved Generalization**: Models trained on diverse features are more likely to perform consistently on new, unseen data.

### 4. Algorithm Selection

We continue using **Logistic Regression** for this multi-feature classification problem.

Logistic regression extends naturally to multiple features by computing a weighted combination of all inputs:

$$
z = w_1 x_1 + w_2 x_2 + \cdots + w_n x_n + b
$$

This linear combination is then passed through the sigmoid function to produce a probability:

$$
P(y = 1 \mid X) = \frac{1}{1 + e^{-z}}
$$

Each feature is assigned its own weight (coefficient), enabling the model to learn the relative importance of each predictor.

---

### 5. Loss Function

The optimization objective remains **Binary Cross-Entropy (Log Loss)**:

$$
L = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log(p_i) + (1 - y_i)\log(1 - p_i) \right]
$$

The model minimizes this loss to determine the optimal weights for all features.

---

### 6. Model Training

The logistic regression model will be trained using all available features. Feature scaling with **StandardScaler** is essential because:

- Features have varying ranges (e.g., Age: 29-77 vs ST depression: 0-6.2)
- Scaling ensures all features contribute proportionally to the model
- It accelerates the convergence process during training

In [ ]:
# Build and train the multi-feature logistic regression model
model_multi = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=2000, random_state=42))
])

# Fit the model to training data
model_multi.fit(X_train_multi, y_train_multi)
print("Multi-feature model training complete!")
print(f"\nTotal features used: {len(X_multi.columns)}")

In [ ]:
# Display the learned model parameters
lr_multi = model_multi.named_steps["lr"]

print("Feature Coefficients (Weights):\n")
feature_importance = pd.DataFrame({
    'Feature': X_multi.columns,
    'Coefficient': lr_multi.coef_[0]
}).sort_values(by='Coefficient', key=abs, ascending=False)

print(feature_importance.to_string(index=False))
print(f"\nIntercept (bias term): {lr_multi.intercept_[0]:.4f}")

#### Understanding Feature Coefficients

- **Positive coefficients** suggest that higher values of that feature are linked to increased heart disease risk
- **Negative coefficients** indicate that higher values correspond to reduced probability of heart disease
- **Coefficient magnitude** reflects the relative influence of each feature (after standardization)

Features with larger absolute coefficient values have greater impact on the prediction outcome.

### 7. Model Evaluation

The multi-feature model is evaluated on the test set using the same metrics as Task 1:

- **Accuracy**: Overall correctness of predictions
- **Precision**: Reliability of positive predictions
- **Recall**: Ability to identify actual heart disease cases
- **F1-Score**: Balanced measure combining precision and recall
- **Confusion Matrix**: Detailed breakdown of prediction outcomes

In [ ]:
# Generate predictions on the test set
y_pred_multi = model_multi.predict(X_test_multi)
print("Predictions generated for multi-feature model")
print("First 10 predictions:", y_pred_multi[:10])
print("First 10 actual values:", y_test_multi.values[:10])

In [ ]:
# Compute evaluation metrics
acc_multi = accuracy_score(y_test_multi, y_pred_multi)
prec_multi = precision_score(y_test_multi, y_pred_multi, zero_division=0)
rec_multi = recall_score(y_test_multi, y_pred_multi, zero_division=0)
f1_multi = f1_score(y_test_multi, y_pred_multi, zero_division=0)
cm_multi = confusion_matrix(y_test_multi, y_pred_multi)

print("="*50)
print("MULTI-FEATURE MODEL RESULTS")
print("="*50)
print(f"Accuracy : {acc_multi:.4f}")
print(f"Precision: {prec_multi:.4f}")
print(f"Recall   : {rec_multi:.4f}")
print(f"F1-score : {f1_multi:.4f}")
print("\nConfusion Matrix:")
print(cm_multi)

In [ ]:
# Detailed classification report
print("\n" + "="*50)
print("DETAILED CLASSIFICATION REPORT")
print("="*50)
print(classification_report(y_test_multi, y_pred_multi, 
                          target_names=['No Heart Disease', 'Heart Disease'],
                          zero_division=0))

In [ ]:
# Sigmoid-style curve for multi-feature logistic regression
# Vary Cholesterol while keeping other features fixed at their mean values

# Create a range of Cholesterol values
chol_min = data["Cholesterol"].min()
chol_max = data["Cholesterol"].max()
chol_range = np.linspace(chol_min, chol_max, 400)

# Base feature set with all features at their mean values
X_base = data.drop(columns=["Heart Disease"]).copy()
mean_values = X_base.mean()

# Create input data for prediction - all features at mean, vary only Cholesterol
X_plot = pd.DataFrame([mean_values] * len(chol_range))
X_plot["Cholesterol"] = chol_range

# Predict probabilities using the multi-feature model
y_prob_multi = model_multi.predict_proba(X_plot)[:, 1]

# Plot sigmoid curve
plt.figure(figsize=(8, 5))
plt.plot(chol_range, y_prob_multi, linewidth=2, color='#2E86AB')
plt.xlabel("Cholesterol", fontsize=12)
plt.ylabel("Predicted Probability of Heart Disease", fontsize=12)
plt.title("Sigmoid Curve: Multi-Feature Logistic Regression\n(All other features held at mean values)", fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## Comparison: Task 1 vs Task 2

Now we compare the performance of the **single-feature model** (Cholesterol only) against the **multi-feature model** (all features) to understand how using multiple predictors affects model performance.

In [ ]:
# Recalculate Task 1 metrics for comparison
y_pred_single = model.predict(X_test)
acc_single = accuracy_score(y_test, y_pred_single)
prec_single = precision_score(y_test, y_pred_single, zero_division=0)
rec_single = recall_score(y_test, y_pred_single, zero_division=0)
f1_single = f1_score(y_test, y_pred_single, zero_division=0)

# Create comparison DataFrame
comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Single Feature (Cholesterol)': [acc_single, prec_single, rec_single, f1_single],
    'Multi-Feature (All)': [acc_multi, prec_multi, rec_multi, f1_multi],
    'Improvement': [
        acc_multi - acc_single,
        prec_multi - prec_single,
        rec_multi - rec_single,
        f1_multi - f1_single
    ]
})

print("="*70)
print("MODEL PERFORMANCE COMPARISON")
print("="*70)
print(comparison.to_string(index=False))
print("="*70)

In [ ]:
# Visualize the comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
single_feature_scores = [acc_single, prec_single, rec_single, f1_single]
multi_feature_scores = [acc_multi, prec_multi, rec_multi, f1_multi]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, single_feature_scores, width, label='Single Feature (Cholesterol)', alpha=0.8)
bars2 = ax.bar(x + width/2, multi_feature_scores, width, label='Multi-Feature (All)', alpha=0.8)

ax.set_xlabel('Metrics', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison: Single vs Multi-Feature', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

### Key Findings

1. **Performance Gains**: The multi-feature model demonstrates notable improvement across all evaluation metrics compared to the single-feature approach. This validates the importance of using comprehensive patient data.

2. **Improved Recall**: The multi-feature model is more effective at detecting actual heart disease cases, which is crucial in medical diagnosis to minimize missed diagnoses (false negatives).

3. **Enhanced Precision**: The model produces more reliable positive predictions, thereby reducing unnecessary false alarms.

4. **Feature Contributions**: Analysis of the coefficients reveals which features have the strongest influence on heart disease prediction.

5. **Clinical Relevance**: In real-world healthcare settings, utilizing comprehensive patient information leads to more accurate and dependable predictions than relying on a single indicator.

---

## Analysis: Comparison of Task 1 and Task 2 Models

### 1. Which Model Performs Better and Why?

The **multi-feature model (Task 2) outperforms** the single-feature model (Task 1) across all evaluation metrics.

**Primary Reasons:**

1. **Richer Information**: The multi-feature model leverages 13 different health indicators (age, blood pressure, cholesterol, chest pain type, etc.) compared to just one (cholesterol). This comprehensive approach enables the model to identify complex relationships among various risk factors.

2. **Superior Pattern Recognition**: Heart disease is a multifactorial condition. While cholesterol is significant, it alone cannot capture the complete picture. Factors such as age, exercise-induced angina, maximum heart rate, and chest pain characteristics together provide a more thorough understanding of cardiovascular health.

3. **Decreased Prediction Bias**: A single-feature model may perform well for patients where that specific feature is dominant but fails for others. The multi-feature model distributes decision-making across multiple indicators, reducing dependence on any single factor.

4. **Capturing Interaction Effects**: The multi-feature model can learn how different features interact. For instance, elevated cholesterol might be more concerning in older patients or those with specific chest pain types. The single-feature model cannot capture these interactions.

5. **Empirical Evidence**: The quantitative improvements shown in the comparison demonstrate that incorporating additional features substantially enhances the model's ability to correctly identify heart disease cases while minimizing false predictions.

---

### 2. How Does Adding More Features Affect Accuracy and Recall?

Incorporating additional features has a **significant positive effect** on both accuracy and recall:

#### Effect on Accuracy:
- **Definition**: Accuracy measures the proportion of correct predictions (both positive and negative) out of all predictions made.
- **Outcome**: The multi-feature model achieves higher accuracy because it can make better-informed decisions using multiple health indicators rather than depending solely on cholesterol levels.
- **Practical Impact**: More patients are correctly classified, whether they have heart disease or not, resulting in fewer overall misdiagnoses.

#### Effect on Recall (Sensitivity):
- **Definition**: Recall measures the proportion of actual heart disease cases that the model correctly identifies.
- **Outcome**: The improvement in recall is especially important in medical diagnosis. The multi-feature model detects more true positive cases, meaning fewer patients with heart disease are incorrectly classified as healthy.
- **Practical Impact**: This reduces **false negatives** (missed diagnoses), which is critical in healthcare where failing to detect heart disease can have life-threatening consequences.

#### Reasons for These Improvements:

1. **Reduced Ambiguity**: A single feature may produce unclear signals. For example, a patient might have moderate cholesterol but still have heart disease due to other risk factors like age, high blood pressure, or chest pain symptoms. The multi-feature model can identify these cases.

2. **Complementary Information**: Different features capture different aspects of cardiovascular health. When one feature is inconclusive, others can provide clarifying information.

3. **Refined Decision Boundaries**: With multiple features, the model can create more sophisticated decision boundaries that separate heart disease and non-heart disease cases more effectively.

4. **Resilience to Outliers**: If one feature has an unusual value for a patient, the model can rely on other features to make a correct prediction, rather than being misled by a single anomalous reading.

---

### 3. Trade-offs Between Interpretability and Performance

There is a clear **trade-off between interpretability and performance** when comparing the two models:

#### Single-Feature Model: High Interpretability, Lower Performance

**Strengths:**
- **Straightforward to Understand**: "Higher cholesterol increases heart disease risk" is a simple, intuitive message that patients and non-technical stakeholders can immediately grasp.
- **Easy to Communicate**: Healthcare providers can easily explain the model's reasoning to patients: "Your cholesterol level suggests X% probability of heart disease."
- **Simple to Validate**: It's straightforward to verify whether the relationship aligns with medical knowledge.
- **Rapid Decision Making**: Only one test result needs to be reviewed.
- **Transparent**: The sigmoid curve clearly illustrates how predictions change with cholesterol levels.

**Weaknesses:**
- **Lower Accuracy**: Ignores important information from other health indicators.
- **Oversimplification**: Heart disease is multifactorial; relying on one feature overlooks this complexity.
- **Higher Misdiagnosis Risk**: More false negatives and false positives.

---

#### Multi-Feature Model: Lower Interpretability, Higher Performance

**Strengths:**
- **Superior Predictions**: Higher accuracy, precision, and recall across all metrics.
- **Holistic View**: Considers the full spectrum of cardiovascular risk factors.
- **Fewer Missed Cases**: Better recall means fewer patients with heart disease are overlooked.
- **More Dependable**: Reduced false positives means fewer unnecessary treatments or patient anxiety.
- **Clinically Realistic**: Aligns with how doctors actually assess heart disease (using multiple indicators).

**Weaknesses:**
- **More Difficult to Explain**: "Your prediction is based on a weighted combination of 13 features" is more complex to communicate.
- **Black Box Perception**: Patients may not understand why they received a particular prediction.
- **Requires Complete Data**: Need information for all 13 features.
- **Complex Feature Interactions**: Understanding how features combine is not straightforward.
- **Higher Cognitive Load**: Medical professionals need to consider multiple coefficients when interpreting results.

---

**Recommendation:**
In a real clinical setting, the **multi-feature model should be preferred** because:
1. Patient safety (fewer missed diagnoses) outweighs ease of explanation
2. Logistic regression remains relatively interpretable compared to deep learning models
3. Feature importance can be communicated to aid understanding
4. The performance improvement justifies the additional complexity

---

## Discussion

This laboratory exercise provided practical experience with logistic regression for binary classification, specifically applied to heart disease prediction. Through implementing both single-feature and multi-feature models, we gained valuable insights into the machine learning pipeline and the significance of feature selection.

The study demonstrates that multi-feature logistic regression models perform substantially better than single-feature models for heart disease prediction, reinforcing the understanding that heart disease is a multifactorial condition. While cholesterol alone provides a basic baseline, incorporating additional clinical features significantly improves accuracy, precision, recall, and F1-score, with recall being the most clinically important improvement as it reduces missed diagnoses.

The results highlight the importance of a well-structured machine learning pipeline, including data cleaning, proper feature scaling, informed feature selection, and comprehensive evaluation using multiple metrics. Model visualizations and coefficient analysis enhance interpretability and align with established medical knowledge, making logistic regression suitable for real-time clinical decision support.

However, challenges such as class imbalance, feature scaling requirements, and the trade-off between interpretability and performance remain. The study is limited to binary classification and a single algorithm type, suggesting that future work should explore advanced models, feature interactions, cross-validation, calibration, and external validation to improve robustness and generalizability.

---

## Conclusion

This laboratory successfully demonstrated the implementation and evaluation of logistic regression models for heart disease prediction, comparing single-feature and multi-feature approaches. The findings clearly indicate that incorporating multiple features significantly enhances model performance across all evaluation metrics, with the multi-feature model achieving higher accuracy, precision, recall, and F1-score.

Following a structured machine learning pipeline—from data collection and cleaning to feature engineering, model training, and comprehensive evaluation—was essential for obtaining reliable results. The improved recall of the multi-feature model is clinically significant, as it reduces false negatives and helps identify more actual heart disease cases, highlighting the potential of ML to support medical decision-making while maintaining patient safety.

Logistic regression provides interpretability through coefficient analysis and offers probabilistic outputs, balancing performance with transparency, computational efficiency, and suitability for real-time clinical application. This lab also emphasized broader lessons: the importance of proper feature engineering, the value of multiple evaluation metrics, ethical considerations in healthcare ML, and the necessity of interdisciplinary collaboration.

Overall, even a relatively straightforward model like logistic regression can deliver substantial clinical insights when applied thoughtfully, reinforcing that effective machine learning is as much about responsible application as it is about algorithmic sophistication.